# W1b: NASA downloads that need your Earthdata login

Survey Crops, plan section W1. Run the cells **top to bottom**; cells 4 to 10 are independent, so you can skip one.
Your Earthdata username and password are typed into cell 2 (the password box is hidden). They are never saved to a file and never printed.

| Cell | Dataset | Who gets it | Time (rough guess, not measured) |
|---|---|---|---|
| 4 | SMAP L4 soil moisture (SPL4SMGP v8) | 64 districts | 10 to 60 min (AppEEARS queue) |
| 5 | MODIS MOD13Q1 NDVI / EVI | 64 districts | **long**: one year first, then all 24 |
| 6 | NASADEM elevation + slope | 64 districts and 2 demo tiles | 20 to 60 min |
| 7 | GRACE-FO JPL mascons | 64 districts (flagged regional) | 5 min |
| 8 | HLS (Landsat + Sentinel-2) NDVI | 2 demo tiles | 10 to 60 min |
| 9 | ECOSTRESS evapotranspiration + land-surface temperature | 2 demo tiles | 10 to 60 min |
| 10 | GEDI L4B biomass (1 km) | 2 demo tiles, context only | downloads about 0.5 GB |

**Before you start:** on your PC run `python scripts/pack_w1b.py` (makes `data/raw/w1b_pack.zip`, 0.4 MB). Cell 1 asks you to upload it once.
**Results** go to Google Drive, folder `cropshift_w1b`. Only the small CSV files are meant for git; copy them to `data/processed/`.

In [ ]:
# 1. Setup: install, mount Drive, unpack the project helpers
!pip -q install earthaccess xarray netCDF4 h5netcdf shapely rasterio
import os, sys, zipfile
from google.colab import drive, files
drive.mount('/content/drive')
WORK = '/content/w1b_work'                      # big temporary downloads (deleted as we go)
OUT = '/content/drive/MyDrive/cropshift_w1b'    # small results kept on Drive
os.makedirs(WORK, exist_ok=True); os.makedirs(OUT, exist_ok=True)
PACK = OUT + '/w1b_pack.zip'
if not os.path.exists(PACK):
    print('Choose data/raw/w1b_pack.zip from your PC')
    up = files.upload()
    open(PACK, 'wb').write(next(iter(up.values())))
zipfile.ZipFile(PACK).extractall('/content/cropshift')
sys.path[:0] = ['/content/cropshift/src/compute', '/content/cropshift/src/acquire']
os.chdir('/content/cropshift')
print('ready; results will be saved in', OUT)

In [ ]:
# 2. Earthdata login (password box is hidden; nothing is saved)
import getpass, requests, earthaccess
from w1b import appeears, products, tables, convert, dem
USER = input('Earthdata username: ')
PW = getpass.getpass('Earthdata password: ')
S = requests.Session()
TOKEN = appeears.login(S, USER, PW)             # AppEEARS bearer token, kept in memory only
os.environ['EARTHDATA_USERNAME'], os.environ['EARTHDATA_PASSWORD'] = USER, PW
auth = earthaccess.login(strategy='environment')  # used by the GRACE-FO and GEDI downloads
os.environ.pop('EARTHDATA_PASSWORD'); del PW
print('AppEEARS login ok; earthaccess authenticated:', bool(auth.authenticated))

In [ ]:
# 3. Common 0.1 degree grid, district weights, and small helpers used by every cell below
import json, shutil, warnings
import numpy as np, pandas as pd, xarray as xr
from district_grid.grid import imerg_axes
from district_grid.boundaries import load_districts
from district_grid.regrid import bin_mean_to_grid
glons, glats = imerg_axes()
BOX = (float(glons[0] - 0.05), float(glats[0] - 0.05), float(glons[-1] + 0.05), float(glats[-1] + 0.05))
DISTRICTS = load_districts()
SLUGS, W = tables.all_district_weights(DISTRICTS)
TILES = products.demo_field_bboxes('/content/cropshift/data/processed')   # the two 20 km demo tiles
print('Bangladesh box', BOX, '| districts', len(SLUGS), '| demo tiles', {k: [round(x, 3) for x in v] for k, v in TILES.items()})

def run_task(task, tag):
    """Submit an AppEEARS task, wait for it, download its .nc files; returns their paths."""
    tid = appeears.submit(S, TOKEN, task)
    print(f'[{tag}] submitted {tid}; waiting (the AppEEARS queue can take a while)')
    appeears.wait(S, TOKEN, tid, poll_s=30)
    paths = [appeears.download(S, TOKEN, tid, fid, f'{WORK}/{tag}/{name}')
             for fid, name, size in appeears.list_files(S, TOKEN, tid)]
    print(f'[{tag}] downloaded', [os.path.basename(p) for p in paths])
    return paths

def axes(ds):
    names = list(ds.variables)
    return ds[convert.find_coord(names, 'lon')].values, ds[convert.find_coord(names, 'lat')].values

def layer(ds, name, sl=slice(None)):
    """One layer as float [time, lat, lon]; `sl` picks a time slice so big files load in chunks."""
    v = ds[convert.find_variable(list(ds.data_vars), name)]
    if 'time' not in v.dims:
        v = v.expand_dims('time')
    v = v.isel(time=sl).transpose('time', convert.find_coord(list(ds.variables), 'lat'),
                                  convert.find_coord(list(ds.variables), 'lon'))
    return v.values.astype(float)

def dates(ds, fmt='%Y-%m-%d'):
    return list(pd.to_datetime(ds['time'].values).strftime(fmt))

def save(df, name):
    path = f'{OUT}/{name}'
    df.to_csv(path, index=False)
    print('saved', name, f'{os.path.getsize(path) / 1e6:.2f} MB', '(over 5 MB: do not put in git)' if os.path.getsize(path) > 5e6 else '')
    return path

In [ ]:
# 4. SMAP L4 (SPL4SMGP v8), 9 km -> mean in each 0.1 degree cell -> 64 district polygons
task = products.area_task('cropshift_smap', 'smap', '2015-03-31', '2026-08-31', BOX)
ds = xr.open_dataset(run_task(task, 'smap')[0])
print(ds)                                    # have a look once: variable names, dates
lons, lats = axes(ds); d = dates(ds)
tabs = {}
for col, lay in [('sm_rootzone_m3m3', 'sm_rootzone'), ('sm_surface_m3m3', 'sm_surface')]:
    a = convert.mask_range(layer(ds, lay), *convert.SMAP_VALID)
    cells, _ = bin_mean_to_grid(a, lons, lats, glons, glats)
    tabs[col] = tables.stack_to_district_table(cells, d, SLUGS, W, col)
    save(tables.to_wide(tabs[col], col).reset_index(), f'smap_{col}_district_daily.csv.gz')
save(tables.coverage_summary(tabs['sm_rootzone_m3m3']), 'smap_district_coverage.csv')
# cross-check against the five point files we already have (same product, one pixel instead of a polygon)
for slug in ['feni', 'cumilla', 'noakhali', 'brahmanbaria', 'sylhet']:
    old = pd.read_csv(f'/content/cropshift/data/processed/smap_{slug}_2015_2026.csv', index_col='date')['sm_rootzone']
    new = tabs['sm_rootzone_m3m3'].query('district == @slug').set_index('date')['sm_rootzone_m3m3']
    j = pd.concat([old.rename('old'), new.rename('new')], axis=1, join='inner').dropna()
    print(slug, 'days', len(j), '| r =', round(j.corr().iloc[0, 1], 3), '| mean(new - old) =', round((j.new - j.old).mean(), 4))
ds.close(); shutil.rmtree(f'{WORK}/smap', ignore_errors=True)

In [ ]:
# 5. MODIS MOD13Q1 NDVI / EVI (250 m, 16-day) -> 0.1 degree cells -> districts. One year at a time, resumable.
YEARS = [2023]        # SMOKE TEST FIRST. When the numbers look right, change to: list(range(2001, 2025))
for y in YEARS:
    name = f'modis_district_{y}.csv.gz'
    if os.path.exists(f'{OUT}/{name}'):
        print(y, 'already done'); continue
    s, e = products.year_chunks(y, y)[0]
    ds = xr.open_dataset(run_task(products.area_task(f'cropshift_modis_{y}', 'modis', s, e, BOX), f'modis{y}')[0])
    if y == YEARS[0]: print(ds)
    lons, lats = axes(ds); d = dates(ds)
    cube = {'ndvi': [], 'evi': []}; shares = []
    for i in range(0, len(d), 4):                      # 4 composites at a time keeps memory small
        sl = slice(i, i + 4)
        rel = layer(ds, 'pixel_reliability', sl)
        for col, lay in [('ndvi', 'NDVI'), ('evi', 'EVI')]:
            clean, share = convert.mod13_clean(layer(ds, lay, sl), rel, name=col)
            cube[col].append(bin_mean_to_grid(clean, lons, lats, glons, glats)[0]); shares.append(share)
    print(y, 'share of pixels kept (good data only):', round(float(np.mean(shares)), 3))
    out = None
    for col in ('ndvi', 'evi'):
        t = tables.stack_to_district_table(np.concatenate(cube[col]), d, SLUGS, W, col)
        out = t if out is None else out.merge(t, on=['date', 'district'], suffixes=('', '_evi'))
    save(out, name)
    ds.close(); shutil.rmtree(f'{WORK}/modis{y}', ignore_errors=True)
# when all years are done, join them:
# parts = [pd.read_csv(f'{OUT}/modis_district_{y}.csv.gz') for y in range(2001, 2025)]; save(pd.concat(parts), 'modis_district_16day_2001_2024.csv.gz')

In [ ]:
# 6. NASADEM 30 m -> mean elevation and mean slope per 0.1 degree cell -> districts; 30 m crops of the 2 demo tiles
task = products.area_task('cropshift_nasadem', 'nasadem', '2000-02-11', '2000-02-21', BOX)
ds = xr.open_dataset(run_task(task, 'nasadem')[0]); print(ds)
lat_n, lon_n = convert.find_coord(list(ds.variables), 'lat'), convert.find_coord(list(ds.variables), 'lon')
hgt = ds[convert.find_variable(list(ds.data_vars), 'NASADEM_HGT')]
if 'time' in hgt.dims: hgt = hgt.isel(time=0, drop=True)
hgt = hgt.transpose(lat_n, lon_n)
LON, LAT = ds[lon_n].values, ds[lat_n].values
STEP = float(abs(LON[1] - LON[0]))
def read_strip(a, b):
    idx = np.where((LAT >= a) & (LAT <= b))[0]
    if idx.size == 0: return np.empty((0, 0)), LON, LAT[:0]
    i0, i1 = int(idx.min()), int(idx.max()) + 1
    z = hgt.isel({lat_n: slice(i0, i1)}).values.astype(float)
    z[z <= -32767] = np.nan
    return z, LON, LAT[i0:i1]
elev, slope, count = dem.dem_cell_stats(read_strip, glons, glats, STEP)
tab = tables.static_to_district_table({'elev_mean_m': elev, 'slope_mean_deg': slope}, SLUGS, W)
print(tab.describe().loc[['min', 'mean', 'max']].round(2))
save(tab, 'nasadem_district_static.csv')
np.savez_compressed(f'{OUT}/nasadem_cells_0p1deg.npz', lons=glons, lats=glats, elev_mean_m=elev, slope_mean_deg=slope, pixel_count=count)
for name, (w_, s_, e_, n_) in TILES.items():       # field-detail tier: keep the 30 m pixels of each demo tile
    ii = np.where((LAT >= s_) & (LAT <= n_))[0]; jj = np.where((LON >= w_) & (LON <= e_))[0]
    z = hgt.isel({lat_n: slice(int(ii.min()), int(ii.max()) + 1), lon_n: slice(int(jj.min()), int(jj.max()) + 1)}).values
    np.savez_compressed(f'{OUT}/nasadem_30m_{name}.npz', elev_m=np.nan_to_num(z, nan=-32768).astype('int16'),
                        lats=LAT[int(ii.min()):int(ii.max()) + 1], lons=LON[int(jj.min()):int(jj.max()) + 1])
    print('saved 30 m crop for', name, z.shape)
ds.close(); shutil.rmtree(f'{WORK}/nasadem', ignore_errors=True)

In [ ]:
# 7. GRACE-FO JPL mascons (PO.DAAC): nearest 0.5 degree cell to each district point, flagged "regional"
key = products.PRODUCTS['grace']
res = earthaccess.search_data(short_name=key['product'])
print(len(res), 'granule(s) found for', key['product'])
path = earthaccess.download(res[:1], f'{WORK}/grace')[0]      # one global file, about 44 MB
ds = xr.open_dataset(path); print(ds)
names = list(ds.variables)
lon_n, lat_n = convert.find_coord(names, 'lon'), convert.find_coord(names, 'lat')
lwe_n = convert.find_variable(list(ds.data_vars), 'lwe_thickness')
unc_n = convert.find_variable(list(ds.data_vars), 'uncertainty')
print('units:', ds[lwe_n].attrs.get('units'), '| long name:', ds[lwe_n].attrs.get('long_name'))
lwe = ds[lwe_n].transpose('time', lat_n, lon_n).values
unc = ds[unc_n].transpose('time', lat_n, lon_n).values
pts = {d['slug']: (d['point'].x, d['point'].y) for d in DISTRICTS}
t = tables.grace_table(lwe, dates(ds), ds[lon_n].values, ds[lat_n].values, pts, uncertainty=unc)
print(t.groupby(['cell_lon', 'cell_lat']).district.nunique().rename('districts_sharing_cell').reset_index())
save(t, 'grace_district_monthly.csv')

In [ ]:
# 8. HLS (Landsat HLSL30 + Sentinel-2 HLSS30) -> NDVI for the 2 demo tiles (cloud, shadow and snow masked)
rows = []
for name, bbox in TILES.items():
    for key, nir in [('hls_l30', 'B05'), ('hls_s30', 'B8A')]:
        task = products.area_task(f'cropshift_{key}_{name}', key, products.DEMO_START, products.DEMO_END, bbox)
        ds = xr.open_dataset(run_task(task, f'{key}_{name}')[0]); d = dates(ds)
        red, nirv, fm = layer(ds, 'B04'), layer(ds, nir), layer(ds, 'Fmask')
        print(name, key, 'Fmask values seen:', sorted(np.unique(fm[np.isfinite(fm)]).astype(int).tolist())[:20])
        nd = convert.hls_ndvi(red, nirv, fm)
        rows.append(tables.scene_stats(nd, d, 'tile_mean_ndvi', tile=name, sensor=key))
        lons, lats = axes(ds)
        np.savez_compressed(f'{OUT}/hls_ndvi_{key}_{name}.npz', dates=np.array(d), lons=lons, lats=lats,
                            ndvi_x10000=np.where(np.isfinite(nd), np.round(nd * 1e4), -9999).astype('int16'))
        ds.close(); shutil.rmtree(f'{WORK}/{key}_{name}', ignore_errors=True)
hls = pd.concat(rows).sort_values(['tile', 'date']); print(hls.groupby(['tile', 'sensor']).clear_share.describe().round(2))
save(hls, 'hls_ndvi_tile_series.csv')

In [ ]:
# 9. ECOSTRESS (70 m): daily evapotranspiration and land-surface temperature for the 2 demo tiles
rows = []
for name, bbox in TILES.items():
    for key, lay, col, fn in [('eco_et', 'ETdaily', 'et_mm_day', convert.wm2_to_mm_per_day),
                              ('eco_lst', 'LST', 'lst_c', convert.kelvin_to_celsius)]:
        task = products.area_task(f'cropshift_{key}_{name}', key, '2018-07-10', '2026-08-31', bbox)
        ds = xr.open_dataset(run_task(task, f'{key}_{name}')[0]); d = dates(ds, '%Y-%m-%dT%H:%M')
        a = layer(ds, lay); cloud = layer(ds, 'cloud')
        a = fn(np.where(cloud == 1, np.nan, a))
        s = tables.scene_stats(a, d, col, tile=name, product=key)
        rows.append(s[s.clear_share > 0]); print(name, key, len(d), 'scenes,', int((s.clear_share > 0).sum()), 'with clear pixels')
        ds.close(); shutil.rmtree(f'{WORK}/{key}_{name}', ignore_errors=True)
eco = pd.concat(rows).sort_values(['product', 'tile', 'date'])
save(eco, 'ecostress_tile_series.csv')

In [ ]:
# 10. GEDI L4B gridded biomass (1 km). CONTEXT ONLY: it mostly measures trees, not field crops.
import rasterio
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds, transform
from district_grid.regrid import bin_mean_scattered
key = products.PRODUCTS['gedi']
res = earthaccess.search_data(concept_id=key['cmr_concept'])
mu = [g for g in res if g['umm']['GranuleUR'].endswith('_MU.tif')]       # mean aboveground biomass density
print(len(res), 'granules;', len(mu), 'mean-biomass file(s). Downloading one file, about 0.5 GB.')
path = earthaccess.download(mu[:1], f'{WORK}/gedi')[0]
with rasterio.open(path) as src:
    print('crs', src.crs, '| pixel size', src.res, '| nodata', src.nodata); print('tags:', src.tags(), src.tags(1))
    win = from_bounds(*transform_bounds('EPSG:4326', src.crs, *BOX), transform=src.transform).round_offsets().round_lengths()
    arr = src.read(1, window=win).astype(float)
    tr = src.window_transform(win); nodata = src.nodata
    cols, rws = np.meshgrid(np.arange(arr.shape[1]) + 0.5, np.arange(arr.shape[0]) + 0.5)
    xs, ys = tr * (cols, rws)
    lon, lat = transform(src.crs, 'EPSG:4326', xs.ravel().tolist(), ys.ravel().tolist())
lon, lat = np.array(lon).reshape(arr.shape), np.array(lat).reshape(arr.shape)
cells, cnt = dem.gedi_cell_means(arr, lon, lat, glons, glats, nodata=nodata)
np.savez_compressed(f'{OUT}/gedi_agbd_cells_0p1deg.npz', lons=glons, lats=glats, agbd_mean=cells, pixel_count=cnt)
v = np.where(arr == nodata, np.nan, arr); rows = []
for name, (w_, s_, e_, n_) in TILES.items():
    m = (lon >= w_) & (lon <= e_) & (lat >= s_) & (lat <= n_) & np.isfinite(v)
    rows.append(dict(tile=name, n_pixels=int(m.sum()), agbd_mean=float(v[m].mean()) if m.any() else np.nan,
                     note=key['caveat'], source=key['source_url']))
save(pd.DataFrame(rows), 'gedi_tile_means.csv'); print(pd.DataFrame(rows)[['tile', 'n_pixels', 'agbd_mean']])
shutil.rmtree(f'{WORK}/gedi', ignore_errors=True)

In [ ]:
# 11. Wrap up: what is on Drive. Copy the small CSV files to data/processed/ on your PC (nothing over 5 MB in git).
for f in sorted(os.listdir(OUT)):
    size = os.path.getsize(f'{OUT}/{f}')
    print(f'{size / 1e6:8.2f} MB  {f}', '   <- raw/large: keep out of git' if size > 5e6 or f.endswith('.npz') else '')
os.environ.pop('EARTHDATA_USERNAME', None)